# ResNet18 last-block fine-tuning — controlled ablation

Bu notebook, tamamlanmış head-only ResNet18 deneyinin validation ile seçilen epoch-9 `best_model.pth` ağırlıklarından yeni bir deney başlatır. Yalnız `layer4` ve 26-sınıflı classifier eğitilir; eski optimizer/history taşınmaz. Aynı frozen split kullanılır ve eski run asla değiştirilmez.

Bu bir klinik tanı sistemi veya dokunulmamış yeni test değildir. Aynı benchmark daha önce iki model için görüldüğünden sonuç kontrollü bir ablation olarak raporlanacaktır.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, time, zipfile

assert 'google.colab' in sys.modules, 'Bu notebook Google Colab için hazırlanmıştır.'
from google.colab import drive
drive.mount('/content/drive')
print('Drive bağlandı.')

## 1. Proje, veri, split ve başlangıç checkpoint'i

Drive ZIP'i Colab'ın geçici diskine açılır. Önceki ResNet18 `best_model.pth` dosyası yalnız başlangıç ağırlığıdır; yeni output klasörü ayrıdır.

In [ ]:
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
ZIP_PATH = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer.zip')
LOCAL_DATA_BASE = Path('/content/NNGA_DATA')
EXTRACT_MARKER = LOCAL_DATA_BASE / '.multi_cancer_extract_complete'
BASELINE_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_26leaf_baseline_v1')
INIT_CHECKPOINT = BASELINE_DIR / 'best_model.pth'
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_lastblock_finetune_v1')
LOG_PATH = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_lastblock_finetune_v1_console.log')
REVIEW_ZIP = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_lastblock_finetune_v1_review_package.zip')
FROZEN_SPLIT_SHA256 = '999bf7a9bd1bba9f2957a9db0e13ae72adb3ecb8fdbc1f33cce40b083c1cf171'
FROZEN_INIT_SHA256 = '20613069067825ea0dfa30e2242678b863cd0acd7b9be3c5b88f265d39285177'
FROZEN_TRAINER_SHA256 = '65de201f74d9296b81280fbd7b4653542a05e6446de6eb987171eb3fa3a72650'
EXPECTED_GROUPS = ('ALL', 'Brain Cancer', 'Breast Cancer', 'Cervical Cancer', 'Kidney Cancer', 'Lung and Colon Cancer', 'Lymphoma', 'Oral Cancer')
EXPECTED_CLASS_NAMES = ('all_benign', 'all_early', 'all_pre', 'all_pro', 'brain_glioma', 'brain_menin', 'brain_tumor', 'breast_benign', 'breast_malignant', 'cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi', 'colon_aca', 'colon_bnt', 'kidney_normal', 'kidney_tumor', 'lung_aca', 'lung_bnt', 'lung_scc', 'lymph_cll', 'lymph_fl', 'lymph_mcl', 'oral_normal', 'oral_scc')

if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
trainer_hash = hashlib.sha256((PROJECT_ROOT / 'src/train.py').read_bytes()).hexdigest()
assert trainer_hash == FROZEN_TRAINER_SHA256, f'Trainer SHA-256 uyuşmuyor: {trainer_hash}'
assert INIT_CHECKPOINT.is_file(), f'Önceki best checkpoint bulunamadı: {INIT_CHECKPOINT}'
init_hash = hashlib.sha256(INIT_CHECKPOINT.read_bytes()).hexdigest()
assert init_hash == FROZEN_INIT_SHA256, f'Başlangıç checkpoint SHA-256 uyuşmuyor: {init_hash}'
assert ZIP_PATH.is_file(), f'Dataset ZIP bulunamadı: {ZIP_PATH}'
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert free_gib >= 12.0, 'Dataset açma ve eğitim için en az 12 GiB yerel boş alan gerekli.'
if not EXTRACT_MARKER.is_file():
    LOCAL_DATA_BASE.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print('ZIP Colab diskine çıkarılıyor; birkaç dakika çıktı görünmeyebilir...')
    subprocess.run(['unzip', '-q', '-o', str(ZIP_PATH), '-d', str(LOCAL_DATA_BASE)], check=True)
    EXTRACT_MARKER.write_text('complete\n', encoding='utf-8')
    print(f'ZIP çıkarma tamamlandı: {(time.monotonic() - started) / 60:.1f} dakika')
dataset_candidates = (LOCAL_DATA_BASE / 'Multi Cancer' / 'Multi Cancer', LOCAL_DATA_BASE / 'Multi Cancer', LOCAL_DATA_BASE)
DATASET_ROOT = next((p for p in dataset_candidates if all((p / g).is_dir() for g in EXPECTED_GROUPS)), None)
assert DATASET_ROOT is not None, 'Sekiz ana klasörü içeren dataset kökü bulunamadı.'
SPLIT_MANIFEST = PROJECT_ROOT / 'research/splits/practical_leakage_reduced_split_v1.csv'
split_hash = hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest()
assert split_hash == FROZEN_SPLIT_SHA256, f'Split SHA-256 uyuşmuyor: {split_hash}'
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('Git commit:', commit)
print('Trainer SHA-256:', trainer_hash)
print('Başlangıç checkpoint SHA-256:', init_hash)
print('Dataset:', DATASET_ROOT)
print('Split SHA-256:', split_hash)
print('Yeni output:', OUTPUT_DIR)

## 2. GPU ve trainable-layer ön kontrolü

T4 GPU seçilmiş olmalı. Bu hücre eğitim yapmaz; gerçek trainer ile yalnız `layer4 + fc` parametrelerinin açık olduğunu doğrular.

In [ ]:
import torch, torchvision
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train import build_model, validate_initial_checkpoint
assert torch.cuda.is_available(), 'GPU seçilmemiş; T4 runtime aç.'
checkpoint, checked_hash = validate_initial_checkpoint(INIT_CHECKPOINT, FROZEN_INIT_SHA256, 'resnet18', 'subclass', EXPECTED_CLASS_NAMES)
assert checkpoint['class_names'] == list(EXPECTED_CLASS_NAMES)
probe = build_model('resnet18', 26, pretrained=True, fine_tune=True)
probe.load_state_dict(checkpoint['model_state_dict'], strict=True)
trainable = {name: p.numel() for name, p in probe.named_parameters() if p.requires_grad}
assert trainable and all(name.startswith(('layer4.', 'fc.')) for name in trainable)
assert sum(trainable.values()) == 8407066, sum(trainable.values())
del probe, checkpoint
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Trainable parameter:', sum(trainable.values()))
print('Açık katmanlar: yalnız layer4 + fc')

## 3. Dondurulmuş ablation sözleşmesi

- Başlangıç: önceki head-only ResNet18 epoch-9 best checkpoint
- Eğitilen kısım: `layer4 + fc`; daha erken katmanlar donuk
- Hedef/split: aynı 26 leaf, aynı 91.000/19.500/19.500 manifest
- AdamW: LR 0.0001, weight decay 0.0001
- En fazla 12 epoch, validation macro-F1 patience 4, seed 42
- Checkpoint seçimi yalnız validation macro-F1; test seçim yapmaz

Tahmini T4 süresi yaklaşık 2–4 saattir; garanti değildir. Runtime kesilirse aynı hücre yalnız aynı training contract ile `last_checkpoint.pth` üzerinden devam eder. Tam çalışmayı başlatmadan önce `AUTHORIZE_FULL_FINETUNE = True` yapılmalıdır.

In [ ]:
AUTHORIZE_FULL_FINETUNE = False
assert AUTHORIZE_FULL_FINETUNE, 'Tam fine-tuning kilitli; önce sözleşmeyi Codex ile onayla.'
required_final = ['best_model.pth', 'last_checkpoint.pth', 'history.csv', 'test_metrics.json', 'test_predictions.csv', 'classification_reports.json', 'leaf_confusion_matrix.png', 'parent_confusion_matrix.png', 'learning_curves.png', 'split_manifest.csv', 'run_config.json']
RUN_COMPLETE = all((OUTPUT_DIR / name).is_file() for name in required_final)
assert not RUN_COMPLETE, 'Bu fine-tuning runı zaten tamamlanmış; yeniden çalıştırma.'
RESUME_RUN = (OUTPUT_DIR / 'last_checkpoint.pth').is_file()
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not RESUME_RUN:
    raise RuntimeError('Output klasörü boş değil ve geçerli resume checkpoint yok; Codex ile incele.')
command = [
    sys.executable, '-u', str(PROJECT_ROOT / 'src/train.py'),
    '--data-dir', str(DATASET_ROOT), '--output-dir', str(OUTPUT_DIR),
    '--model', 'resnet18', '--target-level', 'subclass',
    '--split-manifest', str(SPLIT_MANIFEST), '--pretrained', '--fine-tune',
    '--init-checkpoint', str(INIT_CHECKPOINT), '--init-checkpoint-sha256', FROZEN_INIT_SHA256,
    '--epochs', '12', '--early-stopping-patience', '4', '--batch-size', '32',
    '--image-size', '224', '--normalization', 'imagenet', '--augmentation', 'minimal',
    '--learning-rate', '0.0001', '--weight-decay', '0.0001',
    '--num-workers', '4', '--seed', '42', '--experiment-type', 'ablation',
]
if RESUME_RUN:
    command.append('--resume')
    print('Resume checkpoint bulundu; tamamlanan epochlar yeniden çalışmayacak.')
log_mode = 'a' if RESUME_RUN else 'w'
print('Fine-tuning komutu:', ' '.join(command))
with LOG_PATH.open(log_mode, encoding='utf-8') as log_handle:
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line); log_handle.flush()
    return_code = process.wait()
print('RETURN CODE:', return_code)
if return_code != 0:
    raise RuntimeError('Fine-tuning tamamlanamadı. Output klasörünü değiştirme; logu Codex’e gönder.')

## 4. Tamamlanma, kapsam ve review paketi

Bu hücre 19.500 test tahminini, init/split hash'lerini ve final artifact'ları doğrular; sonra büyük model checkpoint'lerini kopyalamadan indirilebilir review ZIP'i üretir.

In [ ]:
missing = [name for name in required_final if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Eksik fine-tuning çıktıları: {missing}'
copied_split_hash = hashlib.sha256((OUTPUT_DIR / 'split_manifest.csv').read_bytes()).hexdigest()
assert copied_split_hash == FROZEN_SPLIT_SHA256
metrics = json.loads((OUTPUT_DIR / 'test_metrics.json').read_text(encoding='utf-8'))
prediction_rows = sum(1 for _ in (OUTPUT_DIR / 'test_predictions.csv').open(encoding='utf-8')) - 1
assert metrics['target_level'] == 'subclass'
assert metrics['experiment_type'] == 'ablation'
assert metrics['split_sizes'] == {'train': 91000, 'val': 19500, 'test': 19500}
assert metrics['initialization']['checkpoint_sha256'] == FROZEN_INIT_SHA256
assert metrics['trainable_parameter_count'] == 8407066
assert prediction_rows == 19500
review_files = ['test_metrics.json', 'history.csv', 'test_predictions.csv', 'classification_reports.json', 'leaf_classification_report.txt', 'parent_classification_report.txt', 'leaf_confusion_matrix.png', 'parent_confusion_matrix.png', 'learning_curves.png', 'run_config.json']
artifact_hashes = {}
for name in review_files:
    path = OUTPUT_DIR / name
    artifact_hashes[name] = {'sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'size_bytes': path.stat().st_size}
artifact_hashes['split_manifest.csv'] = {'sha256': copied_split_hash, 'size_bytes': (OUTPUT_DIR / 'split_manifest.csv').stat().st_size}
artifact_hashes['best_model.pth'] = {'sha256': hashlib.sha256((OUTPUT_DIR / 'best_model.pth').read_bytes()).hexdigest(), 'size_bytes': (OUTPUT_DIR / 'best_model.pth').stat().st_size}
artifact_hashes['last_checkpoint.pth'] = {'sha256': hashlib.sha256((OUTPUT_DIR / 'last_checkpoint.pth').read_bytes()).hexdigest(), 'size_bytes': (OUTPUT_DIR / 'last_checkpoint.pth').stat().st_size}
artifact_hashes[LOG_PATH.name] = {'sha256': hashlib.sha256(LOG_PATH.read_bytes()).hexdigest(), 'size_bytes': LOG_PATH.stat().st_size}
manifest_path = OUTPUT_DIR / 'artifact_hashes.json'
manifest_path.write_text(json.dumps(artifact_hashes, indent=2), encoding='utf-8')
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_files:
        archive.write(OUTPUT_DIR / name, arcname=name)
    archive.write(manifest_path, arcname='artifact_hashes.json')
    archive.write(LOG_PATH, arcname=LOG_PATH.name)
review_hash = hashlib.sha256(REVIEW_ZIP.read_bytes()).hexdigest()
print('Durum: resnet18_lastblock_finetune_v1_complete')
print('Best epoch:', metrics['best_epoch'])
print('Leaf metrics:', metrics['leaf_metrics'])
print('Parent metrics:', metrics['parent_metrics'])
print('Prediction rows:', prediction_rows)
print('Init SHA-256:', metrics['initialization']['checkpoint_sha256'])
print('Split SHA-256:', copied_split_hash)
print('Review package:', REVIEW_ZIP)
print('Review package SHA-256:', review_hash)